In [4]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import os
import re

def populate_data_table(filepath: str) -> pd.DataFrame:
    """
    Populates a data table by merging CSV files in the specified directory.

    Args:
        filepath (str): The path to the directory containing the CSV files.

    Returns:
        pd.DataFrame: The merged data table, or None if no files are found.
    """
    print("Populating data table from files in", filepath)
    data_table = None  # Initialize data_table outside the loop
    # Divide all columns but the first by the second column
    for file in tqdm(os.listdir(filepath), desc="Processing files", unit="file"):
        df = pd.read_csv(filepath + "/" + file, index_col="OA")

        if data_table is None:
            data_table = df  # Initialize data_table with the first file encountered
        else:
            # Merge on index
            data_table = data_table.merge(df, left_index=True, right_index=True, how="outer")
    return data_table

In [5]:
eng_census_raw = populate_data_table("../data/census_data/eng_raw_csvs")
eng_census_raw = eng_census_raw.reindex(sorted(eng_census_raw.columns), axis=1)

eng_census_raw = eng_census_raw.drop(columns=['ts0060001']) #density column
# Remove the Wales only variables
strings_to_remove = ['ts032', 'ts033', 'ts034', 'ts035', 'ts036', 'ts076']
columns_to_keep = [col for col in eng_census_raw.columns if not any(string in col for string in strings_to_remove)]
eng_census_raw = eng_census_raw[columns_to_keep]

# Extract unique prefixes by removing the last four digits
prefixes = set(re.sub(r"\d{4}$", "", col) for col in eng_census_raw.columns if re.match(r".*\d{4}$", col))
# Normalize each group
for prefix in prefixes:
    base_col = f"{prefix}0001"  # The assumed total column
    group_cols = [col for col in eng_census_raw.columns if col.startswith(prefix)]

    if base_col in eng_census_raw.columns:  # Ensure the base column exists
        eng_census_raw[group_cols] = eng_census_raw[group_cols].div(eng_census_raw[base_col], axis=0)

# Filter columns that end with '001' (the total columns)
columns_to_drop = [col for col in eng_census_raw.columns if col.endswith('001')]
eng_census_raw = eng_census_raw.drop(columns=columns_to_drop)
#round to 3dp 
eng_census_raw = eng_census_raw.round(3)

#drop ts020 and ts055 (they are variables for small subsets (non-uk residents and second homes) so do not have values for most OAs
columns_to_drop = [col for col in eng_census_raw.columns if col.startswith('ts020') or col.startswith('ts055')]
eng_census_raw = eng_census_raw.drop(columns=columns_to_drop)

#print columns with nans
print("Columns with missing values:")
#in eng_census_raw
missing_columns = eng_census_raw.columns[eng_census_raw.isnull().any()].tolist()
print(missing_columns)

Populating data table from files in ../data/census_data/eng_raw_csvs


Processing files: 100%|██████████| 52/52 [00:26<00:00,  1.94file/s]


Columns with missing values:
[]


In [6]:
#Save the cleaned data (reset index so that OA is a column)
eng_census_raw.reset_index().to_parquet("../data/census_data/engcensus_cleaned_scaled.parquet", index=False)
#save the zscored data
eng_census_raw_z = eng_census_raw.copy()
#zscore the data
scaler = StandardScaler()
eng_census_raw_zstandardized = scaler.fit_transform(eng_census_raw_z)
eng_census_raw_zstandardized = pd.DataFrame(eng_census_raw_zstandardized, columns=eng_census_raw.columns, index=eng_census_raw.index)
#save the zscored data
eng_census_raw_zstandardized.reset_index().to_parquet("../data/census_data/engcensus_cleaned_zstandardized.parquet", index=False)


In [7]:
def get_pca_filter_columns(All_census):

    # Filter columns with 'ts' in their names.
    ts_columns = [col for col in All_census.columns if 'ts' in col]
    ts_data = All_census[ts_columns]

    # Standardize the data.
    scaler = StandardScaler()
    ts_data_std = scaler.fit_transform(ts_data)

    # Apply PCA to account for 80% of the variance.
    pca = PCA(n_components=0.8)  # Adjusted to select components for 80% variance.
    pca.fit(ts_data_std)

    # The explained variance ratio indicates the importance of each principal component.
    explained_variance_ratio = pca.explained_variance_ratio_

    # Number of components chosen.
    num_components_chosen = pca.n_components_

    # Components_ gives the loadings of each variable on the principal components.
    loadings = pca.components_

    # Calculate the overall importance of each variable within the chosen components.
    # This is done by summing the squares of the loadings across the chosen components for each variable.
    overall_importance = np.sum(loadings**2, axis=0)

    # Identify the indices of the most important variables based on overall importance within the chosen components.
    important_indices = np.argsort(overall_importance)[::-1][:min(200, num_components_chosen)]

    # Extract the names of the most important variables.
    important_variables = np.array(ts_columns)[important_indices]

    return important_variables

def get_high_variance_columns(All_census,var_threshold=1.2):
    variances = {col: All_census[col].var() for col in All_census if col.startswith("ts")}

# Store variances in a new DataFrame
    var_df = pd.DataFrame(list(variances.items()), columns=['Column', 'Variance'])

# Find the median variance
#median_variance = All_census.var().median()  # this raises an error
    median_variance = var_df['Variance'].median() # I assume this is the aim, based on the comment and code above

# Determine 20% higher than the median variance
    threshold_variance = median_variance * var_threshold

# Subset the original DataFrame to include columns 20% higher than the median variance
    selected_columns = [col for col, var in variances.items() if var > threshold_variance]

    return selected_columns


In [8]:
#run PCA and varience prefilter
pca_columns = get_pca_filter_columns(eng_census_raw)
var_columns = get_high_variance_columns(eng_census_raw)
#Select all the "important" variables based on PCA and varience
concatenated_columns = np.unique(np.concatenate((pca_columns, var_columns)))
subset_df = eng_census_raw[concatenated_columns]
# Save the scaled DataFrame to a CSV file
subset_df.reset_index().to_parquet("../data/census_data/engcensus_cleaned_scaled_filtered.parquet", index=False)
# Save the zstandardized DataFrame to a CSV file
subset_df_z = eng_census_raw_zstandardized[concatenated_columns]
subset_df_z.reset_index().to_parquet("../data/census_data/engcensus_cleaned_zstandardized_filtered.parquet", index=False)
